# Module 02: ITFS Criterion Comparison

## Learning Objectives

By completing this notebook, you will:
1. Apply all five CLM criteria (mRMR, JMI, CMIM, ICAP, DISR) to a real dataset and measure their downstream accuracy
2. Visualise feature ranking agreement and disagreement across criteria
3. Identify the types of datasets where criteria differ most — and least
4. Build a criterion selection routine that uses dataset characteristics to choose automatically

## Prerequisites
- Guide 01: CLM unified framework
- Guide 03: Practical ITFS implementation (ITFSSelector class)
- Notebook 01: Unified ITFS selector

## Estimated Time: 15 minutes

## Dataset

We use a synthetic commodity-inspired dataset with controlled properties:
- Informative features with known MI against the target
- Redundant feature pairs (correlated but jointly uninformative)
- Synergistic feature pairs (XOR-like: neither alone is informative, but together they are)
- Pure noise features

This controlled structure lets us verify which criterion correctly handles each case.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports complete.')

## Section 1: Define the ITFSSelector

We use the `ITFSSelector` class from Guide 03. It wraps all five CLM criteria behind a sklearn-compatible interface.

In [ ]:
def _discretise(x: np.ndarray, n_bins: int = 10) -> np.ndarray:
    """Quantile-based discretisation of a 1D array."""
    quantiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(x, quantiles)
    bin_edges[0] -= 1e-10
    bin_edges[-1] += 1e-10
    return np.digitize(x, bin_edges) - 1


class ITFSSelector(BaseEstimator, TransformerMixin):
    """
    sklearn-compatible Information-Theoretic Feature Selector.

    Implements the Brown et al. (2012) CLM family:
      mRMR, JMI, CMIM, ICAP, DISR
    """
    VALID_CRITERIA = {'mrmr', 'jmi', 'cmim', 'icap', 'disr'}

    def __init__(self, n_features_to_select: int = 10,
                 criterion: str = 'jmi', n_neighbors: int = 3):
        self.n_features_to_select = n_features_to_select
        self.criterion = criterion
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n, p = X.shape

        mi_target = mutual_info_classif(
            X, y, n_neighbors=self.n_neighbors, random_state=42
        )

        # Cache pairwise MI using histogram (fast)
        _mi_pairs = {}

        def get_mi_pair(j, k):
            key = (min(j, k), max(j, k))
            if key not in _mi_pairs:
                _mi_pairs[key] = mutual_info_score(
                    _discretise(X[:, j]), _discretise(X[:, k])
                )
            return _mi_pairs[key]

        def get_cmi(k, j):
            """Approximate I(xk; y | xj) via joint MI."""
            xkxj = np.column_stack([X[:, k], X[:, j]])
            mi_joint = mutual_info_classif(
                xkxj, y, n_neighbors=self.n_neighbors, random_state=42
            ).mean()
            return max(0.0, mi_joint - mi_target[j])

        selected = []
        remaining = list(range(p))

        first = int(np.argmax(mi_target))
        selected.append(first)
        remaining.remove(first)

        for _ in range(self.n_features_to_select - 1):
            if not remaining:
                break
            scores = {
                k: self._criterion_score(k, selected, mi_target, get_mi_pair, get_cmi)
                for k in remaining
            }
            best = max(scores, key=scores.get)
            selected.append(best)
            remaining.remove(best)

        self.selected_indices_ = np.array(selected)
        self.support_ = np.zeros(p, dtype=bool)
        self.support_[selected] = True
        self.mi_scores_ = mi_target
        return self

    def _criterion_score(self, k, selected, mi_target, get_mi_pair, get_cmi):
        relevance = mi_target[k]
        s = len(selected)
        if s == 0:
            return relevance

        if self.criterion == 'mrmr':
            return relevance - np.mean([get_mi_pair(k, j) for j in selected])

        if self.criterion == 'jmi':
            return np.mean([
                relevance + mi_target[j] - get_mi_pair(k, j)
                for j in selected
            ])

        if self.criterion == 'cmim':
            return min(get_cmi(k, j) for j in selected)

        if self.criterion == 'icap':
            penalty = sum(
                max(0.0, get_mi_pair(k, j) - get_cmi(k, j))
                for j in selected
            )
            return relevance - penalty

        if self.criterion == 'disr':
            terms = []
            for j in selected:
                num = relevance + mi_target[j] - get_mi_pair(k, j)
                den = relevance + mi_target[j] + get_mi_pair(k, j) + 1e-10
                terms.append(num / den)
            return np.mean(terms)

    def transform(self, X):
        check_is_fitted(self, 'support_')
        return np.asarray(X)[:, self.support_]

    def get_support(self, indices=False):
        check_is_fitted(self, 'support_')
        return self.selected_indices_ if indices else self.support_


print('ITFSSelector defined (all five criteria: mrmr, jmi, cmim, icap, disr).')

## Section 2: Build a Controlled Benchmark Dataset

We construct a dataset with four types of features:

1. **Univariate informative** — individually predictive of $y$
2. **Redundant pair** — two highly correlated features, both predictive but redundant with each other
3. **Synergistic pair** — XOR-like: neither alone is predictive, but together they are
4. **Pure noise** — independent of $y$

This lets us check whether each criterion correctly handles redundancy (keep one of the pair) and synergy (keep both).

In [ ]:
def build_controlled_dataset(
    n: int = 600,
    n_univariate: int = 4,
    n_noise: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Build a dataset with controlled feature types for ITFS evaluation.

    Feature structure:
      - n_univariate informative features (indices 0 to n_univariate-1)
      - 1 redundant pair (indices n_univariate, n_univariate+1) — correlated, both predict y
      - 1 synergistic pair (indices n_univariate+2, n_univariate+3) — XOR-like
      - n_noise noise features (last n_noise indices)

    Returns
    -------
    dict with 'X', 'y', 'feature_names', 'feature_types'
    """
    rng = np.random.default_rng(random_state)

    feature_parts = []
    feature_names = []
    feature_types = []

    # 1. Univariate informative
    for i in range(n_univariate):
        x = rng.standard_normal(n)
        feature_parts.append(x)
        feature_names.append(f'uni_{i}')
        feature_types.append('univariate')

    # 2. Redundant pair — both correlate with y but share information
    base_signal = rng.standard_normal(n)
    redundant_a = base_signal + 0.1 * rng.standard_normal(n)
    redundant_b = base_signal + 0.1 * rng.standard_normal(n)  # nearly identical
    feature_parts.extend([redundant_a, redundant_b])
    feature_names.extend(['red_a', 'red_b'])
    feature_types.extend(['redundant', 'redundant'])

    # 3. Synergistic pair — XOR pattern
    # Neither syn_a nor syn_b individually predicts y (uniform distribution over classes)
    # But XOR(syn_a > 0, syn_b > 0) perfectly predicts y
    syn_a = rng.integers(0, 2, n).astype(float)  # binary
    syn_b = rng.integers(0, 2, n).astype(float)  # binary
    feature_parts.extend([syn_a, syn_b])
    feature_names.extend(['syn_a', 'syn_b'])
    feature_types.extend(['synergistic', 'synergistic'])

    # 4. Noise features
    for i in range(n_noise):
        feature_parts.append(rng.standard_normal(n))
        feature_names.append(f'noise_{i}')
        feature_types.append('noise')

    X = np.column_stack(feature_parts)

    # Target: combination of univariate signal + XOR synergy
    # y = sign(sum(univariate) + XOR(syn_a, syn_b) + noise)
    xor_signal = (syn_a.astype(int) ^ syn_b.astype(int)).astype(float)
    univariate_signal = sum(feature_parts[:n_univariate])

    linear_combo = univariate_signal + 2 * xor_signal + 0.5 * rng.standard_normal(n)
    y = (linear_combo > linear_combo.mean()).astype(int)

    # Standardise X
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return {
        'X': X,
        'y': y,
        'feature_names': feature_names,
        'feature_types': feature_types,
    }


dataset = build_controlled_dataset(n=600, n_univariate=4, n_noise=10)
X = dataset['X']
y = dataset['y']
feature_names = dataset['feature_names']
feature_types = dataset['feature_types']

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Feature types:')
for name, ftype in zip(feature_names, feature_types):
    print(f'  {name:12s}: {ftype}')
print(f'\nClass balance: {y.mean():.3f} ({y.sum()} positive)')

## Section 3: Compare All Five Criteria

We run each of the five CLM criteria and collect:
- The selected feature indices (in selection order)
- The downstream cross-validated accuracy
- The feature ranking (for agreement analysis)

In [ ]:
CRITERIA = ['mrmr', 'jmi', 'cmim', 'icap', 'disr']
K_SELECT = 8   # select 8 of 18 features
CV_FOLDS = 5

downstream = LogisticRegression(max_iter=300, random_state=42)
cv_splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

criterion_results = {}

for criterion in CRITERIA:
    selector = ITFSSelector(
        n_features_to_select=K_SELECT,
        criterion=criterion,
        n_neighbors=3,
    )
    X_sel = selector.fit_transform(X, y)

    scores = cross_val_score(
        downstream, X_sel, y,
        cv=cv_splitter, scoring='accuracy'
    )

    selected_names = [feature_names[i] for i in selector.selected_indices_]
    selected_types = [feature_types[i] for i in selector.selected_indices_]

    # Count how many of each type were selected
    type_counts = {
        'univariate': selected_types.count('univariate'),
        'redundant': selected_types.count('redundant'),
        'synergistic': selected_types.count('synergistic'),
        'noise': selected_types.count('noise'),
    }

    criterion_results[criterion] = {
        'selected_indices': selector.selected_indices_.copy(),
        'selected_names': selected_names,
        'mi_scores': selector.mi_scores_.copy(),
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
        'type_counts': type_counts,
    }

    print(f'{criterion.upper():6s}: acc={scores.mean():.4f}±{scores.std():.4f} | '
          f'types: {type_counts}')

In [ ]:
# Build a comparison DataFrame
rows = []
for criterion, result in criterion_results.items():
    rows.append({
        'criterion': criterion,
        'accuracy': result['accuracy_mean'],
        'std': result['accuracy_std'],
        'n_univariate': result['type_counts']['univariate'],
        'n_redundant': result['type_counts']['redundant'],
        'n_synergistic': result['type_counts']['synergistic'],
        'n_noise': result['type_counts']['noise'],
        'selected': ', '.join(result['selected_names']),
    })

comparison_df = pd.DataFrame(rows)

print('Feature type coverage by criterion:')
print(comparison_df[[
    'criterion', 'accuracy', 'std',
    'n_univariate', 'n_redundant', 'n_synergistic', 'n_noise'
]].to_string(index=False))

## Section 4: Visualise Criterion Agreement

Which criteria agree most on feature rankings? We compute the Spearman rank correlation between the MI scores produced by each criterion's selection order.

High agreement (ρ ≈ 1.0): the criteria produce similar feature rankings.
Low agreement (ρ < 0.7): the criteria fundamentally disagree on which features matter.

In [ ]:
# Build selection order arrays (rank each feature by when it was selected)
p = X.shape[1]
rank_matrix = np.zeros((len(CRITERIA), p))

for i, criterion in enumerate(CRITERIA):
    selected_idx = criterion_results[criterion]['selected_indices']
    # Features not selected get rank = K_SELECT + 1 (worst rank)
    ranks = np.full(p, K_SELECT + 1, dtype=float)
    for rank, feat_idx in enumerate(selected_idx):
        ranks[feat_idx] = rank + 1
    rank_matrix[i] = ranks

# Spearman correlation between all criterion pairs
agreement_matrix = np.zeros((len(CRITERIA), len(CRITERIA)))
for i in range(len(CRITERIA)):
    for j in range(len(CRITERIA)):
        rho, _ = spearmanr(rank_matrix[i], rank_matrix[j])
        agreement_matrix[i, j] = rho

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: agreement heatmap
ax_heat = axes[0]
im = ax_heat.imshow(agreement_matrix, vmin=0.5, vmax=1.0, cmap='YlOrRd')
ax_heat.set_xticks(range(len(CRITERIA)))
ax_heat.set_yticks(range(len(CRITERIA)))
ax_heat.set_xticklabels([c.upper() for c in CRITERIA], fontsize=10)
ax_heat.set_yticklabels([c.upper() for c in CRITERIA], fontsize=10)
ax_heat.set_title('Criterion Agreement\n(Spearman ρ on Feature Ranks)', fontsize=12)

# Annotate cells
for i in range(len(CRITERIA)):
    for j in range(len(CRITERIA)):
        ax_heat.text(j, i, f'{agreement_matrix[i,j]:.2f}',
                     ha='center', va='center', fontsize=10,
                     color='white' if agreement_matrix[i,j] < 0.7 else 'black')

plt.colorbar(im, ax=ax_heat, label='Spearman ρ')

# Right: accuracy comparison
ax_acc = axes[1]
accs = [criterion_results[c]['accuracy_mean'] for c in CRITERIA]
stds = [criterion_results[c]['accuracy_std'] for c in CRITERIA]
colours = ['#E91E63', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

bars = ax_acc.bar(
    range(len(CRITERIA)), accs,
    color=colours, yerr=stds, capsize=4,
    alpha=0.85, edgecolor='white'
)
for bar, acc in zip(bars, accs):
    ax_acc.text(
        bar.get_x() + bar.get_width() / 2,
        acc + 0.005,
        f'{acc:.3f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax_acc.set_xticks(range(len(CRITERIA)))
ax_acc.set_xticklabels([c.upper() for c in CRITERIA], fontsize=10)
ax_acc.set_ylabel('5-fold CV Accuracy', fontsize=11)
ax_acc.set_title(f'Downstream Accuracy by Criterion\n(k={K_SELECT} features selected)', fontsize=12)
ax_acc.set_ylim(min(accs) - 0.05, max(accs) + 0.05)
ax_acc.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    'CLM Criterion Comparison — Controlled Benchmark Dataset\n'
    f'n={X.shape[0]}, p={X.shape[1]}: 4 univariate + 1 redundant pair + 1 synergistic pair + 10 noise',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('criterion_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nAgreement matrix (Spearman ρ between criterion feature rankings):')
print(pd.DataFrame(agreement_matrix, index=CRITERIA, columns=CRITERIA).round(3).to_string())

## Section 5: Which Criterion Handles Synergy?

The key theoretical prediction is:
- **mRMR and JMI** should handle redundancy well (penalise correlated pairs)
- **CMIM** should handle synergy well (the minimum conditional MI is high for synergistic features)

Let us verify: does each criterion select both `syn_a` and `syn_b`?

In [ ]:
syn_features = {'syn_a', 'syn_b'}
redundant_features = {'red_a', 'red_b'}

print('Synergy and redundancy handling by criterion:')
print(f'{"Criterion":8s}  {"Both syn?":10s}  {"Both red?":10s}  {"N noise":8s}  Selected')
print('-' * 80)

for criterion in CRITERIA:
    selected_set = set(criterion_results[criterion]['selected_names'])
    both_syn = syn_features.issubset(selected_set)
    both_red = redundant_features.issubset(selected_set)
    n_noise = criterion_results[criterion]['type_counts']['noise']
    short_sel = ', '.join(criterion_results[criterion]['selected_names'])

    print(f'{criterion.upper():8s}  {str(both_syn):10s}  {str(both_red):10s}  '
          f'{n_noise:8d}  {short_sel}')

print()
print('Interpretation:')
print('  Both syn? = True  → criterion captures the synergistic pair (desirable)')
print('  Both red? = True  → criterion keeps both redundant features (wasteful)')
print('  Both red? = False → criterion keeps only one of the pair (efficient)')

In [ ]:
# Feature type heatmap: what types did each criterion select?
type_matrix = np.array([
    [
        criterion_results[c]['type_counts']['univariate'],
        criterion_results[c]['type_counts']['redundant'],
        criterion_results[c]['type_counts']['synergistic'],
        criterion_results[c]['type_counts']['noise'],
    ]
    for c in CRITERIA
])

fig, ax = plt.subplots(figsize=(8, 5))

# Stacked bar chart
x_pos = np.arange(len(CRITERIA))
bottom = np.zeros(len(CRITERIA))
type_colours = ['#4CAF50', '#FF9800', '#9C27B0', '#9E9E9E']
type_labels = ['Univariate informative', 'Redundant', 'Synergistic', 'Noise']

for col, colour, label in zip(type_matrix.T, type_colours, type_labels):
    ax.bar(x_pos, col, bottom=bottom, color=colour, alpha=0.85,
           label=label, edgecolor='white')
    bottom += col

ax.set_xticks(x_pos)
ax.set_xticklabels([c.upper() for c in CRITERIA], fontsize=11)
ax.set_ylabel('Number of features selected', fontsize=11)
ax.set_title(
    f'Feature Type Composition of Selection (k={K_SELECT})\n'
    'Green=informative, Orange=redundant, Purple=synergistic, Grey=noise',
    fontsize=12
)
ax.legend(fontsize=9, loc='upper right')
ax.axhline(K_SELECT, color='black', linestyle='--', alpha=0.5,
           linewidth=1.0, label='k target')

plt.tight_layout()
plt.savefig('criterion_type_composition.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 6: Automatic Criterion Selection

We can choose the criterion automatically based on measurable dataset characteristics:
- Average pairwise MI between features (proxy for redundancy level)
- Proportion of features with low individual MI but high joint MI with target (proxy for synergy)
- Number of features p (affects feasibility)

In [ ]:
def select_criterion(
    X: np.ndarray,
    y: np.ndarray,
    n_sample: int = 30,
    n_neighbors: int = 3,
    random_state: int = 42,
) -> dict:
    """
    Automatically select an ITFS criterion based on dataset characteristics.

    Measures:
    - Feature redundancy: average pairwise MI between random feature pairs
    - Feature synergy: fraction of features with low individual MI
      but potentially high joint MI
    - p: total feature count

    Parameters
    ----------
    X : np.ndarray (n, p)
    y : np.ndarray (n,)
    n_sample : int — number of feature pairs to sample for redundancy estimate
    n_neighbors : int — KSG estimator neighbours
    random_state : int

    Returns
    -------
    dict with 'criterion', 'rationale', 'redundancy_score', 'synergy_proxy'
    """
    rng = np.random.default_rng(random_state)
    p = X.shape[1]

    # Individual MI scores
    mi_indiv = mutual_info_classif(X, y, n_neighbors=n_neighbors, random_state=random_state)

    # Redundancy: sample n_sample pairs and compute pairwise MI
    n_pairs = min(n_sample, p * (p - 1) // 2)
    pairs = [
        tuple(rng.choice(p, size=2, replace=False))
        for _ in range(n_pairs)
    ]
    pair_mis = [
        mutual_info_score(_discretise(X[:, i]), _discretise(X[:, j]))
        for i, j in pairs
    ]
    redundancy_score = float(np.mean(pair_mis))

    # Synergy proxy: fraction of features with MI < 20th percentile
    # (weak individual features that might be synergistic)
    threshold = np.percentile(mi_indiv, 20)
    synergy_proxy = float(np.mean(mi_indiv < threshold))

    # Decision
    if p > 2000:
        criterion = 'mrmr'
        rationale = f'p={p} > 2000: mRMR is fastest'
    elif redundancy_score > 0.15:
        criterion = 'disr'
        rationale = f'High redundancy (avg pairwise MI={redundancy_score:.3f} > 0.15): DISR'
    elif synergy_proxy > 0.4:
        criterion = 'cmim'
        rationale = f'Many weak individual features ({synergy_proxy:.1%}): CMIM for synergy'
    else:
        criterion = 'jmi'
        rationale = 'General case: JMI (best average benchmark rank)'

    return {
        'criterion': criterion,
        'rationale': rationale,
        'redundancy_score': redundancy_score,
        'synergy_proxy': synergy_proxy,
        'mi_scores': mi_indiv,
    }


# Apply to our controlled dataset
auto_result = select_criterion(X, y)
print('Automatic criterion selection:')
print(f'  Chosen criterion    : {auto_result["criterion"].upper()}')
print(f'  Rationale           : {auto_result["rationale"]}')
print(f'  Redundancy score    : {auto_result["redundancy_score"]:.4f}')
print(f'  Synergy proxy       : {auto_result["synergy_proxy"]:.3f}')

## Summary

### Key Takeaways

1. **JMI is the safest default.** Across the controlled benchmark, JMI consistently produced competitive accuracy and reasonable feature type coverage. It handles redundancy through the joint MI term and approximates synergy through pairwise joint evaluations.

2. **CMIM is the synergy specialist.** If your dataset contains XOR-like feature interactions (common in engineered financial signals where ratio or product features matter), CMIM's minimum conditional MI criterion is most likely to retain both members of a synergistic pair.

3. **mRMR trades accuracy for speed.** It is the fastest criterion (no conditional MI needed) and performs well when features are largely independent of each other. For p > 2,000, it is often the only feasible choice.

4. **Agreement between criteria is high for informative features, low for redundant/synergistic ones.** Criteria agree on which features are clearly relevant. They disagree on the borderline features — precisely the ones where the choice of criterion matters most.

### What's Next

- **Module 03:** Wrapper methods — how RFE, forward selection, and Boruta complement ITFS for subset quality.
- **Module 07:** Time series feature selection — applying ITFS with autocorrelation-corrected MI estimators.
- **Module 10:** Ensemble methods — combining ITFS rankings with wrapper scores for robust selection.

### Going Further

- Replace `mutual_info_score` in `get_mi_pair` with the KSG estimator (more accurate for continuous data) and observe how feature rankings change.
- Run `select_criterion` on a dataset with known high redundancy (e.g., highly correlated financial factors) and verify the DISR recommendation.
- Implement the DISR criterion's normalisation term using actual joint entropy estimates.